# High-Level API - One-Liner ML

TuiML's high-level API lets you **train, evaluate, compare, save, and serve** models with single function calls. No boilerplate, no manual splitting, no metric plumbing.

| Function | Purpose |
|----------|---------|
| `tuiml.train()` | Train and evaluate a model |
| `tuiml.run()` | Execute from a config dict or JSON file |
| `tuiml.experiment()` | Compare algorithms across datasets |
| `tuiml.predict()` | Make predictions with a trained model |
| `tuiml.evaluate()` | Evaluate a model on test data |
| `tuiml.save()` / `load()` | Persist and restore models |
| `tuiml.list_algorithms()` | Discover available algorithms |
| `tuiml.describe_algorithm()` | Get parameter schema for an algorithm |
| `tuiml.search_algorithms()` | Search algorithms by keyword |
| `tuiml.serve()` / `stop_server()` | Serve a model via REST API |

In [1]:
import tuiml
import numpy as np

## 1. `tuiml.train()` - The Core One-Liner

The simplest call needs three things: an **algorithm name**, a **dataset**, and a **target column**.

In [2]:
# Simplest possible call - train on built-in iris dataset
result = tuiml.train("RandomForestClassifier", "iris", "class")

print(type(result))          # WorkflowResult
print("Metrics:", result.metrics)
print("Model:", type(result.model).__name__)

<class 'tuiml.workflow.WorkflowResult'>
Metrics: {'accuracy_score': 0.9666666666666667, 'f1_score': 0.9473684210526316}
Model: RandomForestClassifier


Pass algorithm hyperparameters as keyword arguments directly:

In [3]:
# Algorithm parameters go straight into train() as kwargs
result = tuiml.train(
    "RandomForestClassifier",
    "iris",
    "class",
    n_estimators=100,
    max_depth=10,
)

print("Accuracy:", result.metrics.get("accuracy_score", "N/A"))

Accuracy: 0.9666666666666667


## 2. Cross-Validation

Set `cv=` to use k-fold cross-validation instead of a single train/test split. The returned metrics are averaged across all folds.

In [4]:
# 5-fold cross-validation
result = tuiml.train("NaiveBayesClassifier", "iris", "class", cv=5)

print("5-fold CV metrics:", result.metrics)

5-fold CV metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9405483405483406, 'cv_f1_score_std': 0.03865951411172374}


In [5]:
# 10-fold cross-validation (the standard in ML research)
result = tuiml.train("C45TreeClassifier", "iris", "class", cv=10)

print("10-fold CV metrics:", result.metrics)
print("CV details:", result.cv_results)

10-fold CV metrics: {'cv_accuracy_score_mean': 0.9400000000000001, 'cv_accuracy_score_std': 0.05537749241945382, 'cv_f1_score_mean': 0.9174592074592075, 'cv_f1_score_std': 0.07879977082980924}
CV details: {'scores': {'accuracy_score': [0.9333333333333333, 1.0, 1.0, 0.9333333333333333, 1.0, 0.8666666666666667, 0.8666666666666667, 1.0, 0.9333333333333333, 0.8666666666666667], 'f1_score': [0.9090909090909091, 1.0, 1.0, 0.923076923076923, 1.0, 0.8, 0.8, 1.0, 0.9090909090909091, 0.8333333333333334]}}


## 3. Preprocessing

The `preprocessing=` parameter accepts a list of preprocessor names or dicts with parameters.

In [6]:
# List of preprocessor names (uses default parameters)
result = tuiml.train(
    "SVC",
    "diabetes",
    "class",
    preprocessing=["SimpleImputer", "StandardScaler"],
    cv=5,
)

print("With preprocessing:", result.metrics)

With preprocessing: {'cv_accuracy_score_mean': 0.7604278074866311, 'cv_accuracy_score_std': 0.021367430294661732, 'cv_f1_score_mean': 0.6080089861828992, 'cv_f1_score_std': 0.03600311917894893}


In [7]:
# List of dicts for full control over preprocessor parameters
result = tuiml.train(
    "KNearestNeighborsClassifier",
    "diabetes",
    "class",
    preprocessing=[
        {"name": "SimpleImputer", "strategy": "median"},
        {"name": "MinMaxScaler"},
    ],
    cv=5,
    k=5,
)

print("Custom preprocessing:", result.metrics)

Custom preprocessing: {'cv_accuracy_score_mean': 0.7174857821916646, 'cv_accuracy_score_std': 0.016616235325243874, 'cv_f1_score_mean': 0.5543756066179355, 'cv_f1_score_std': 0.04505069221709752}


## 4. Presets

Presets are named preprocessing bundles. Use `preset=` to apply a full pipeline in one parameter.

| Preset | Steps | Use case |
|--------|-------|----------|
| `"minimal"` | None | Clean data, no preprocessing needed |
| `"fast"` | Impute (most frequent) | Quick baseline with missing value handling |
| `"standard"` | Impute + MinMaxScale + OneHotEncode | General-purpose pipeline |
| `"full"` | Impute + StandardScale + OneHotEncode + SelectKBest | Full pipeline with feature selection |
| `"imbalanced"` | Impute + MinMaxScale + SMOTE | Class-imbalanced datasets |

In [8]:
# Standard preset: Impute + Scale + Encode
result = tuiml.train(
    "LogisticRegression",
    "diabetes",
    "class",
    preset="standard",
    cv=5,
)

print("Standard preset:", result.metrics)

Standard preset: {'cv_accuracy_score_mean': 0.6562770562770562, 'cv_accuracy_score_std': 0.04033852795319463, 'cv_f1_score_mean': 0.5103827200047253, 'cv_f1_score_std': 0.03640929741361372}


In [9]:
# Full preset: includes feature selection
result = tuiml.train(
    "RandomForestClassifier",
    "diabetes",
    "class",
    preset="full",
    cv=5,
)

print("Full preset:", result.metrics)

Full preset: {'cv_accuracy_score_mean': 0.6796791443850267, 'cv_accuracy_score_std': 0.03437356844467985, 'cv_f1_score_mean': 0.22503184560290337, 'cv_f1_score_std': 0.03529373624932178}


In [10]:
# Inspect what each preset contains
for name, config in tuiml.PRESETS.items():
    steps = [s["name"] if isinstance(s, dict) else s for s in config["preprocessing"]]
    fs = config.get("feature_selection")
    fs_name = fs["name"] if fs else "None"
    print(f"{name:12s} -> {steps}  |  feature_selection={fs_name}")

minimal      -> []  |  feature_selection=None
fast         -> ['SimpleImputer']  |  feature_selection=None
standard     -> ['SimpleImputer', 'MinMaxScaler', 'OneHotEncoder']  |  feature_selection=None
full         -> ['SimpleImputer', 'StandardScaler', 'OneHotEncoder']  |  feature_selection=SelectKBestSelector
imbalanced   -> ['SimpleImputer', 'MinMaxScaler', 'SMOTESampler']  |  feature_selection=None


## 5. Feature Selection

Add feature selection with `feature_selection=`. Works alongside preprocessing or presets.

In [11]:
result = tuiml.train(
    "NaiveBayesClassifier",
    "diabetes",
    "class",
    preprocessing=["SimpleImputer", "StandardScaler"],
    feature_selection="SelectKBestSelector",
    cv=5,
)

print("With feature selection:", result.metrics)

With feature selection: {'cv_accuracy_score_mean': 0.7539258127493421, 'cv_accuracy_score_std': 0.01880855772244167, 'cv_f1_score_mean': 0.628075059448524, 'cv_f1_score_std': 0.03425409270946835}


## 6. Algorithm as Dict (LLM-Friendly Format)

Instead of passing the algorithm as a string + kwargs, you can pass a single dictionary. This is particularly useful when an LLM generates the configuration as JSON.

In [12]:
# Algorithm specified as a dict - ideal for LLM-generated configs
result = tuiml.train(
    {"name": "RandomForestClassifier", "n_estimators": 50, "max_depth": 8},
    "iris",
    "class",
    cv=5,
)

print("Dict-based algorithm:", result.metrics)

Dict-based algorithm: {'cv_accuracy_score_mean': 0.9533333333333335, 'cv_accuracy_score_std': 0.02666666666666666, 'cv_f1_score_mean': 0.9315873015873016, 'cv_f1_score_std': 0.040388638581663285}


## 7. `tuiml.run()` - Config-Driven Workflows

`tuiml.run()` takes a full configuration dictionary (or a path to a JSON file) and passes it to `train()`. This is the preferred entry point for automated pipelines and LLM agents.

In [13]:
config = {
    "algorithm": "XGBoostClassifier",
    "data": "iris",
    "target": "class",
    "preprocessing": ["SimpleImputer", "MinMaxScaler"],
    "cv": 5,
    "n_estimators": 50,
}

result = tuiml.run(config)
print("Config-driven result:", result.metrics)

/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:56:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Config-driven result: {'cv_accuracy_score_mean': 0.9466666666666667, 'cv_accuracy_score_std': 0.03399346342395189, 'cv_f1_score_mean': 0.9200083542188805, 'cv_f1_score_std': 0.053842600422922486}


In [15]:
# An LLM might generate this JSON config
llm_config = {
    "algorithm": {"name": "SVC", "kernel": "rbf", "C": 1.0},
    "data": "iris",
    "target": "class",
    "preprocessing": [
        {"name": "SimpleImputer", "strategy": "mean"},
        {"name": "StandardScaler"},
    ],
    "cv": 10,
}

result = tuiml.run(llm_config)
print("LLM config result:", result.metrics)

LLM config result: {'cv_accuracy_score_mean': 0.9666666666666668, 'cv_accuracy_score_std': 0.04472135954999579, 'cv_f1_score_mean': 0.9505244755244755, 'cv_f1_score_std': 0.07623680680305436}


## 8. `tuiml.experiment()` - Compare Algorithms

Compare multiple algorithms across one or more datasets with a single call. Returns an `Experiment` object with summary tables, rankings, and statistical tests.

In [17]:
exp = tuiml.experiment(
    algorithms=[
        "RandomForestClassifier",
        "NaiveBayesClassifier",
        "C45TreeClassifier",
        "SVC",
        "KNearestNeighborsClassifier",
    ],
    datasets=["iris", "diabetes"],
    cv=10,
)

print(exp.summary())

Experiment: Experiment
Validation: cross_validation
Metric: accuracy

Dataset: iris
--------------------------------------------------
  RandomForestClassifier: 0.9533 ± 0.0427
  NaiveBayesClassifier: 0.9533 ± 0.0600
  C45TreeClassifier: 0.9400 ± 0.0467
  SVC: 0.9733 ± 0.0327
  KNearestNeighborsClassifier: 0.9600 ± 0.0442

Dataset: diabetes
--------------------------------------------------
  RandomForestClassifier: 0.7605 ± 0.0472
  NaiveBayesClassifier: 0.7578 ± 0.0378
  C45TreeClassifier: 0.6511 ± 0.0034
  SVC: 0.7591 ± 0.0282
  KNearestNeighborsClassifier: 0.6823 ± 0.0291



In [18]:
# Export as markdown table
print(exp.to_markdown())

| Dataset | C45TreeClassifier | KNearestNeighborsClassifier | NaiveBayesClassifier | RandomForestClassifier | SVC |
|---|---|---|---|---|---|
| iris | 0.9400 ± 0.0492 | 0.9600 ± 0.0466 | 0.9533 ± 0.0632 | 0.9533 ± 0.0450 | 0.9733 ± 0.0344 |
| diabetes | 0.6511 ± 0.0036 | **0.6823 ± 0.0306** ▲ | **0.7578 ± 0.0398** ▲ | **0.7605 ± 0.0498** ▲ | **0.7591 ± 0.0298** ▲ |
|---|---|---|---|---|---|
| **W/L/T** | 0/0/2 | 1/0/1 | 1/0/1 | 1/0/1 | 1/0/1 |


## 9. `tuiml.predict()` & `tuiml.evaluate()`

Use the trained model from any `WorkflowResult` to make predictions or evaluate on new data.

In [19]:
# Train a model
result = tuiml.train("RandomForestClassifier", "iris", "class")

# Predict on new samples
new_data = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.7, 3.0, 5.2, 2.3],
])
predictions = tuiml.predict(result.model, new_data)
print("Predictions:", predictions)

Predictions: [0 2]


In [20]:
# Evaluate against known labels
from tuiml.datasets import load_iris
from tuiml.evaluation import train_test_split

X, y = load_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train on the training set
from tuiml.algorithms.trees import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

# Evaluate on the test set
scores = tuiml.evaluate(model, X_test, y_test)
print("Evaluation:", scores)

Evaluation: {'accuracy_score': 1.0}


## 10. `tuiml.save()` & `tuiml.load()`

Persist trained models to disk and reload them later.

In [21]:
import os

# Train
result = tuiml.train("NaiveBayesClassifier", "iris", "class", cv=5)
print("Original metrics:", result.metrics)

# Save
tuiml.save(result.model, "iris_nb_model.pkl")

# Load
loaded_model = tuiml.load("iris_nb_model.pkl")

# Predict with loaded model
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
print("Loaded model prediction:", tuiml.predict(loaded_model, sample))

# Clean up
os.remove("iris_nb_model.pkl")

Original metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9405483405483406, 'cv_f1_score_std': 0.03865951411172374}
Loaded model prediction: [0]


## 11. Algorithm Discovery

Three functions help you explore what TuiML offers.

In [22]:
# List all classifiers
classifiers = tuiml.list_algorithms(type="classifier")
print(f"{len(classifiers)} classifiers available:")
for algo in classifiers[:8]:  # Show first 8
    print(f"  {algo['name']}")

42 classifiers available:
  NaiveBayesClassifier
  NaiveBayesMultinomialClassifier
  BayesianNetworkClassifier
  DecisionStumpClassifier
  C45TreeClassifier
  RandomTreeClassifier
  RandomForestClassifier
  ReducedErrorPruningTreeClassifier


In [23]:
# Describe a specific algorithm and its parameters
info = tuiml.describe_algorithm("RandomForestClassifier")
print(f"Name: {info['name']}")
print(f"Type: {info['type']}")
print("Parameters:")
for param, schema in info["parameters"].items():
    print(f"  {param}: {schema.get('type', '?')} = {schema.get('default', '?')}")

Name: RandomForestClassifier
Type: ComponentType.CLASSIFIER
Parameters:
  n_estimators: integer = 100
  max_features: ['string', 'integer', 'number'] = sqrt
  max_depth: integer = None
  min_samples_split: integer = 2
  min_samples_leaf: integer = 1
  bootstrap: boolean = True
  oob_score: boolean = False
  random_state: integer = None
  n_jobs: integer = 1


In [24]:
# Search by keyword
results = tuiml.search_algorithms("boost")
print("Algorithms matching 'boost':")
for algo in results:
    print(f"  {algo['name']}")

Algorithms matching 'boost':
  DecisionStumpClassifier
  LogisticModelTreeClassifier
  LogisticRegression
  SimpleLinearRegression
  SimpleLogisticRegression
  BaggingClassifier
  AdaBoostClassifier
  VotingClassifier
  StackingClassifier
  AdditiveRegression
  LogitBoostClassifier
  RandomSubspaceClassifier
  MultiClassClassifier
  XGBoostClassifier
  XGBoostRegressor
  CatBoostClassifier
  CatBoostRegressor
  LightGBMClassifier
  LightGBMRegressor


## 12. API Discovery for LLMs

The `get_api_info()` function returns machine-readable JSON schemas for every high-level function. LLM agents can call this to discover what TuiML can do and what parameters each function accepts.

In [25]:
from tuiml.api import get_api_info

# Get schema for a single function
train_schema = get_api_info("train")
print("train() schema:")
print(f"  Description: {train_schema['description']}")
print(f"  Returns: {train_schema['returns']}")
print(f"  Parameters: {list(train_schema['parameters'].keys())}")

train() schema:
  Description: Train a machine learning model with a complete workflow
  Returns: WorkflowResult with model, metrics, predictions
  Parameters: ['algorithm', 'data', 'target', 'preprocessing', 'feature_selection', 'test_size', 'cv', 'metrics', 'preset']


In [26]:
# Get schemas for all functions (LLM tool discovery)
all_schemas = get_api_info()
print(f"{len(all_schemas)} API functions available:")
for name, schema in all_schemas.items():
    print(f"  {name:22s} - {schema['description']}")

14 API functions available:
  train                  - Train a machine learning model with a complete workflow
  run                    - Execute workflow from configuration dict or JSON file
  pipeline               - Create a fluent workflow for method chaining
  predict                - Make predictions with a trained model
  evaluate               - Evaluate model performance on test data
  experiment             - Compare multiple algorithms on multiple datasets
  list_algorithms        - List available algorithms in registry
  describe_algorithm     - Get detailed info and parameter schema for an algorithm
  search_algorithms      - Search algorithms by keyword
  save                   - Save trained model to disk
  load                   - Load trained model from disk
  serve                  - Serve a trained model via REST API
  stop_server            - Stop running model server(s)
  server_status          - Get status of running model servers
